## 1. Import Libraries & Dependencies

In [95]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
rng = np.random.default_rng(42)

## 2. Load the Dataset

In [ ]:
data_train = pd.read_csv("mnist_train.csv")
data_test = pd.read_csv("mnist_test.csv")

## 3. Convert DataFrames to NumPy Arrays

In [ ]:
data_train = data_train.to_numpy()
data_test = data_test.to_numpy()

In [ ]:
data_train.shape

In [ ]:
data_test.shape

## 5. Feature and Target Splitting ($X$ / $y$ Split)

In [ ]:
X_train = data_train[:,1:]
y_train = data_train[:,0]
X_test = data_test[:,1:]
y_test = data_test[:,0]

## 6. Data Visualization & Sample Inspection

In [ ]:
first_image = data_train[0][1:].reshape(28,28)
data_train[0][0]

In [ ]:
plt.imshow(first_image, cmap='gray')
plt.axis('off')
plt.show()

## resape

In [ ]:
X_train = X_train.reshape(-1,1,28,28)
X_test = X_test.reshape(-1,1,28,28)

In [ ]:
X_train.shape

## Zero-center

In [ ]:
mean_image = np.mean(X_test,axis=0)
X_train = X_train - mean_image
X_test = X_test - mean_image

## ReLU

In [ ]:
def relu(x):
    return np.maximum(0,x)

## Padding

In [1]:
def padding(x,pad = 1):

    num_row = x.shape[2]
    num_column = x.shape[3]

    new_column = num_column + 2*pad
    new_row = num_row + 2*pad

    zeros = np.zeros((x.shape[0],1,new_row,new_column))
    print(zeros.shape)
    for i in range(x.shape[0]):
        zeros[i,0,pad:pad+num_row,pad:pad + num_column] = x[i,0]

    return zeros

In [2]:
def padding_with_numpy(x,pad = 1):
    return np.pad(x,((0,0),(0,0),(pad,pad),(pad,pad)),mode='constant',constant_values=0)

In [3]:
matrix_test = np.arange(24000).reshape(60,1,20,20)

NameError: name 'np' is not defined

In [ ]:
ft = np.array([[0,1,0],
                  [0,1,0],
                  [0,1,0]])


In [ ]:
ft

In [ ]:
matrix_test.shape

### vanilla convolution

In [ ]:
def vanilla_conv(x,filters,stride =1):

    out_row = int(np.floor((x.shape[2]-filters.shape[0])/stride) + 1)
    out_col = int(np.floor((x.shape[3]-filters.shape[0])/stride) + 1)
    empty = np.zeros((x.shape[0],x.shape[1],out_row,out_col))
    for i in range(x.shape[0]):
        for j in range(out_row):
            for k in range(out_col):
                empty[i,0,j,k] = np.einsum('ij,ij->',x[i,0,j:j+filters.shape[0],k:k+filters.shape[0]],filters)

    return empty

 ### vanilla max Pooling

In [ ]:
def vanilla_maxpool(max_pool_input,size_filter = 2 ,stride = 1):

    row_size = max_pool_input.shape[2]
    output_size = int((row_size - size_filter)//stride) +1
    output_maxpool = np.zeros((max_pool_input.shape[0],1,output_size,output_size))

    for i in range(max_pool_input.shape[0]):
        for j in range(output_size):
            for k in range(output_size):
                h_start = j * stride
                v_start = k * stride
                output_maxpool[i,0,j,k] = np.max(max_pool_input[i,0,h_start:h_start+size_filter,v_start:v_start+size_filter])
    return output_maxpool

## im2col

### convolution with im2col

In [ ]:
def convolution(input_conv, filters, stride =1):
    number_of_input = input_conv.shape[0]
    number_of_row = input_conv.shape[2]
    size_filter = filters.shape[0]

    output_size = int((number_of_row - size_filter)//stride + 1)

    temp_matrix = np.zeros((number_of_input , size_filter ** 2 , output_size ** 2))
    print(temp_matrix.shape)

    for i in range( number_of_input):

        for j in range(output_size):
            for k in range(output_size):

                h_start = j * stride
                v_start = k * stride

                temp_matrix[i,:,j * output_size + k] = (input_conv[
                i,
                0,
                h_start : h_start + size_filter,
                v_start : v_start + size_filter ].flatten())

    output_conv = ((np.expand_dims(filters.flatten(),axis=0))@temp_matrix).reshape(number_of_input,1 ,output_size,output_size)


    return output_conv

### max pooling

In [ ]:
def max_pooling(input_conv,size_filter = 2,stride =2):
    number_of_input = input_conv.shape[0]
    number_of_row = input_conv.shape[2]
    size_filter = size_filter

    output_size = int((number_of_row - size_filter)//stride + 1)

    temp_matrix = np.zeros((number_of_input , size_filter ** 2 , output_size ** 2))

    for i in range( number_of_input):
        for j in range(output_size):
            for k in range(output_size):

                h_start = j * stride
                v_start = k * stride

                temp_matrix[i,:,j * output_size + k] = (input_conv[
                i,
                0,
                h_start : h_start + size_filter,
                v_start : v_start + size_filter ].flatten())

    output_maxpool = ((np.max(temp_matrix,axis=1)).reshape(number_of_input,1 ,output_size,output_size))


    return output_maxpool

## avg pooling

In [ ]:
def avg_pooling(input_conv,size_filter = 2,stride =2):
    number_of_input = input_conv.shape[0]
    number_of_row = input_conv.shape[2]
    size_filter = size_filter

    output_size = int((number_of_row - size_filter)//stride + 1)

    temp_matrix = np.zeros((number_of_input , size_filter ** 2 , output_size ** 2))

    for i in range( number_of_input):
        for j in range(output_size):
            for k in range(output_size):

                h_start = j * stride
                v_start = k * stride

                temp_matrix[i,:,j * output_size + k] = (input_conv[
                i,
                0,
                h_start : h_start + size_filter,
                v_start : v_start + size_filter ].flatten())

    output_avg_pooling = ((np.average(temp_matrix,axis=1)).reshape(number_of_input,1 ,output_size,output_size))


    return output_avg_pooling

## activation

In [ ]:
def relu(x):
    return np.maximum(0,x)

In [ ]:
def sigmoid(x):
    return 1/(1+np.exp(-x))

## initialization

In [ ]:
def xavier_init(shape):

    """
    برای لایه دنس: shape = (input_dim, output_dim)
    برای لایه کانو: shape = (num_filters, in_channels, height, width)
    """
    if len(shape) == 2:
        fan_in = shape[0]
    elif len(shape) == 4:
        fan_in = shape[1] * shape[2] * shape[3]
    else:
        raise ValueError("xavier_init_error")

    std = np.sqrt(1 / fan_in)
    return rng.normal(0,std, size = shape)

تله‌ی مخفی نامپای (بحرانی): اگر شبکه عصبی تو در یک تکرار (Epoch) به قدری اشتباه پیش‌بینی کند که احتمال یک کلاس را دقیقاً 0.0 تشخیص دهد، دستور np.log(0) مقدار -inf (منفی بی‌نهایت) را تولید می‌کند! ضرب این مقدار در برچسب‌ها باعث می‌شود کل خروجی لغزش کند و به عدد NaN (مخفف Not a Number) تبدیل شود و کل شبکه عصبی‌ات نابود شود.

In [ ]:
def he_init(shape):
    """
    برای لایه دنس: shape = (input_dim, output_dim)
    برای لایه کانو: shape = (num_filters, in_channels, height, width)
    """
    if len(shape) == 2:
        fan_in = shape[0]
    elif len(shape) == 4:
        fan_in = shape[1] * shape[2] * shape[3]
    else:
        raise ValueError("he_init_error")

    std = np.sqrt(2 / fan_in)
    return rng.normal(0,std,size=shape)


## Cross entropy loss

In [ ]:
def ce_loss(labels,predictions):
    predictions = np.clip(predictions,1e-15,1)
    return - np.sum(labels * np.log(predictions))

# class neural network

In [ ]:
class neural_networks():
    def __init__(self,input_dim,output_dim,layer_sizes,conv_list,activation = sigmoid,initialize_weights = True):
        self.weights = []
        self.biases = []
        self.filter_list = []
        self.input_dim = input_dim
        self.output_dim = output_dim
        self.layer_sizes = layer_sizes
        self.initialize_weights = initialize_weights
        self.activation = activation
        self.conv_list = conv_list

    def initialize_weights(self,flag):
        current_layer = self.input_dim

        for i in range(len(self.layer_sizes)):
            if flag and self.activation == "sigmoid":
                self.weights.append(xavier_init((current_layer , self.layer_sizes[i])))
            elif flag and self.activation == "relu":
                self.weights.append(he_init((current_layer , self.layer_sizes[i])))
            else:
                self.weights.append(rng.normal(0,1,size=(current_layer , self.layer_sizes[i])))

            self.biases.append(np.zeros((self.layer_sizes[i],1)))
            current_layer = self.layer_sizes[i]


    def forward(self,x):

        current_input= x

        if self.activation == "sigmoid": # تعیین نوع اکتیوشن برای اعمال و مقدار دهی های اولی
            activation = sigmoid
        elif self.activation == "relu":
            activation = relu



        for layer in self.conv_list: # تعیین نوع ورودی لیست کانولوشن
            if layer[0] == "maxpool":
                func = max_pooling
            elif layer[0] == "avgpool":
                func = avg_pooling
            elif layer[0] == "conv":
                func = convolution
            #[[conv,size_filter : 3,2],[maxpool,2,2],[],[]]

            z = func(current_input,layer[1],layer[2])

            #نکته معماری: این کار از نظر سینتکس پایتون ارور نمی‌دهد، اما از نظر معماری شبکه‌های عصبی یک ایراد ساختاری است. ما تابع فعال‌سازی (مثل ReLU) را معمولاً بلافاصله بعد از لایه کانو اعمال می‌کنیم. لایه‌های پولینگ (مثل MaxPool یا AvgPool) خودشان صرفاً کار کاهش ابعاد را انجام می‌دهند و دیگر نیازی به تابع فعال‌سازی مجدد ندارند.


            # داشتم  مقدار دهی فیلتر و اعمال فیلتر روی کانو را کامل می کردم ادامه فردا
            if layer[0]=="conv":
                if activation == "sigmoid":
                    self.filter_list.append(xavier_init((number_of_input,number_of_channel,layer[1],layer[1])))

                elif activation == "relu":
                    self.filter_list.append(he_init((number_of_input,number_of_channel,layer[1],layer[1])))

                a = activation(z,self.filter_list[],layer[2])
                current_input = a
            else:
                current_input = z



        current_input = current_input.reshape(-1,self.input_dim) # رابط لایه کانولوشن به لایه دنس

        for i in range(len(self.layer_sizes)-1):

            z = current_input @ self.weights[i] + self.biases[i].T
            a = activation(z)
            current_input = a

        z = current_input @ self.weights[-1]+ self.biases[-1].T
        a = activation(z)
        return a

    def backward(self):
        ...

    def update(self):
        ...

    def train(self):
        ...